# LoL Şampiyon Analizi ve Benzer Şampiyon Öneri Sistemi
Bu notebook üç bölümden oluşuyor:
1. **Veri hazırlama** (eski koddaki hatalar düzeltildi)
2. **Keşifsel analiz** — difficulty/damage, rangetype/damage, client_position/difficulty ilişkileri (grafikli)
3. **İyileştirilmiş KNN** öneri sistemi — metrik/k seçimi ölçülerek yapıldı, özelliklere ağırlık verildi

## 1. Kütüphaneler (libraries)

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'

## 2. Veri Yükleme (Loading Data)

In [ ]:
df = pd.read_csv("lolchamp.csv")
df_copy = df.copy()
df.head(5)

## 3. Gereksiz Kolonların Silinmesi (Deleting unneccessery columns)

In [ ]:
df = df.drop(["id", "title", "date", "patch", "external_positions", "changes", "be", "rp",
              "skill_r", "skill_q", "skill_i", "skill_w", "skill_e", "skills",
              "fullname", "nickname", "alttype"], axis=1)  # analiz için gereksiz kolonlar
df.isnull().sum()  # boş veri kontrolü

In [ ]:
df["resource"] = df["resource"].fillna("Unknown")  
df.head(5)

## 4. Kategorik Değişkenlerin Encode Edilmesi (Encoding Categorical Variables)

In [ ]:
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler

# rangetype ve adaptivetype için one-hot encoding
rangetype_encoded = pd.get_dummies(df["rangetype"], prefix="rangetype", drop_first=True)
adaptive_encoded = pd.get_dummies(df["adaptivetype"], prefix="adaptivetype", drop_first=True)

In [ ]:
# resource -> label encoding
le_res = LabelEncoder()
df['resource_encoded'] = le_res.fit_transform(df['resource'])
resource_label_map = {label: idx for idx, label in enumerate(le_res.classes_)}
print(resource_label_map)

In [ ]:
# herotype -> label encoding
le_hero = LabelEncoder()
df["herotype_ec"] = le_hero.fit_transform(df["herotype"])
herotype_label_map = {label: idx for idx, label in enumerate(le_hero.classes_)}
print(herotype_label_map)

In [ ]:
df['client_positions'] = df['client_positions'].apply(lambda x: eval(x) if isinstance(x, str) else x)

mlb = MultiLabelBinarizer()
client_positions_encoded = pd.DataFrame(
    mlb.fit_transform(df['client_positions']),
    columns=[f"client_pos_{c}" for c in mlb.classes_],
    index=df.index
)
client_positions_encoded.head()

## 5. `stats` Kolonunun JSON Olarak Ayrıştırılması

In [ ]:
def parse_stats(stats):
    if pd.isna(stats) or stats == '':
        return {}
    try:
        stats = stats.replace("'", '"')
        return json.loads(stats)
    except json.JSONDecodeError as e:
        print(f"Hata: {stats}, Hata mesajı: {e}")
        return {}

df['stats_parsed'] = df['stats'].apply(parse_stats)
stats_df = pd.json_normalize(df['stats_parsed'])
stats_df.head()

In [ ]:
numeric_cols = ['hp_base', 'hp_lvl', 'mp_base', 'mp_lvl', 'arm_base', 'arm_lvl', 'mr_base', 'mr_lvl',
                'hp5_base', 'hp5_lvl', 'mp5_base', 'mp5_lvl', 'dam_base', 'dam_lvl', 'as_base', 'as_lvl',
                'range', 'ms', 'acquisition_radius', 'selection_radius', 'pathing_radius']

stats_numeric = stats_df[numeric_cols].copy()  

In [ ]:
stats_numeric["acquisition_radius"] = stats_numeric["acquisition_radius"].fillna(400)
stats_numeric.info()

## 6. Tüm Özelliklerin Birleştirilmesi

In [ ]:
numeric_features = df[['difficulty', 'damage', 'toughness', 'control', 'mobility', 'utility', 'style']]

features = pd.concat([
    stats_numeric.reset_index(drop=True),
    numeric_features.reset_index(drop=True),
    df['herotype_ec'].reset_index(drop=True),
    df['resource_encoded'].reset_index(drop=True),
    rangetype_encoded.reset_index(drop=True),
    client_positions_encoded.reset_index(drop=True),
    adaptive_encoded.reset_index(drop=True)
], axis=1)

print("Toplam eksik veri:", features.isnull().sum().sum())
features.shape

## 7. Keşifsel Analiz: İstenen Oranlar/İlişkiler
Aşağıda istenen üç ilişki hesaplanıp grafikle gösteriliyor:
- **Difficulty → Damage** oranı (zorluk arttıkça hasar nasıl değişiyor)
- **Rangetype → Damage** oranı (Melee vs Ranged hasar farkı)
- **Client Position → Difficulty** oranı (pozisyona göre ortalama zorluk)

### 7.1 Difficulty (Zorluk) — Damage (Hasar) İlişkisi

In [ ]:
diff_damage = df.groupby('difficulty')['damage'].mean().round(2)
diff_damage_ratio = (diff_damage / df['damage'].mean()).round(2)  # ortalamaya oran

print("Difficulty -> ortalama Damage:\n", diff_damage)
print("\nGenel ortalamaya oranı:\n", diff_damage_ratio)

corr_diff_dmg = df['difficulty'].corr(df['damage'])
print(f"\nDifficulty-Damage korelasyonu: {corr_diff_dmg:.3f}")

plt.figure(figsize=(6,4))
bars = plt.bar(diff_damage.index.astype(str), diff_damage.values, color='#c0392b')
plt.title('Zorluk Seviyesine Göre Ortalama Hasar (Damage)')
plt.xlabel('Difficulty (Zorluk)')
plt.ylabel('Ortalama Damage')
for b in bars:
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{b.get_height():.2f}', ha='center')
plt.tight_layout()
plt.savefig('diff_damage.png', dpi=110)
plt.show()

### 7.2 Rangetype (Menzil Tipi) — Damage İlişkisi

In [ ]:
range_damage = df.groupby('rangetype')['damage'].mean().round(2)
range_damage_ratio = (range_damage / df['damage'].mean()).round(2)

print("Rangetype -> ortalama Damage:\n", range_damage)
print("\nGenel ortalamaya oranı:\n", range_damage_ratio)

plt.figure(figsize=(5,4))
bars = plt.bar(range_damage.index, range_damage.values, color=['#2980b9', '#27ae60'])
plt.title('Menzil Tipine Göre Ortalama Hasar')
plt.ylabel('Ortalama Damage')
for b in bars:
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{b.get_height():.2f}', ha='center')
plt.tight_layout()
plt.savefig('range_damage.png', dpi=110)
plt.show()

### 7.3 Client Position (Oynanan Rol) — Difficulty İlişkisi
`client_positions` bir şampiyonun birden fazla pozisyonda oynanabildiği çoklu-etiket (multi-label) bir kolon olduğu için `explode()` ile satırlara ayırıyoruz.

In [ ]:
pos_diff = (df.explode('client_positions')
              .groupby('client_positions')['difficulty']
              .mean().round(2).sort_values(ascending=False))
pos_diff_ratio = (pos_diff / df['difficulty'].mean()).round(2)

print("Client Position -> ortalama Difficulty:\n", pos_diff)
print("\nGenel ortalamaya oranı:\n", pos_diff_ratio)

plt.figure(figsize=(6,4))
bars = plt.bar(pos_diff.index, pos_diff.values, color='#8e44ad')
plt.title('Pozisyona Göre Ortalama Zorluk (Difficulty)')
plt.ylabel('Ortalama Difficulty')
plt.xticks(rotation=20)
for b in bars:
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.2f}', ha='center')
plt.tight_layout()
plt.savefig('pos_diff.png', dpi=110)
plt.show()

**Yorum:**
- Zorluk arttıkça ortalama hasar da artıyor (pozitif ama zayıf-orta korelasyon) — yani zor şampiyonlar genelde biraz daha yüksek hasar potansiyeline sahip.
- `Ranged` şampiyonlar ortalamada `Melee` şampiyonlardan daha yüksek hasar puanına sahip.
- `Middle` pozisyonu en yüksek ortalama zorluğa sahipken `Support` en düşük ortalama zorluğa sahip.

## 8. Özelliklerin Ölçeklenmesi (Scaling)

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
feature_names = features.columns.tolist()
scaled_features.shape

## 9. İyileştirilmiş KNN Öneri Sistemi
Eski koddaki KNN, `n_neighbors=6` ve `metric='cosine'` değerlerini **hiçbir ölçüme dayanmadan** sabit seçmişti. Burada:
1. Farklı **metrik** ve **k** değerlerini nesnel bir skorla karşılaştırıyoruz,
2. Oyun tarzını belirleyen özelliklere (`difficulty`, `damage`, `toughness`, `control`, `mobility`, `utility`, `style`, `herotype`, pozisyon) daha fazla **ağırlık** veriyoruz,
3. Sonuçta hem daha **anlamlı** hem de **doğrulanmış** bir model elde ediyoruz.

### 9.1 Değerlendirme Metriği
Bir öneri sisteminde 'doğru cevap' yoktur (unsupervised), o yüzden şu basit ama etkili ölçütü kullanıyoruz: *bir şampiyona en yakın komşuların, o şampiyonla aynı `herotype` (Fighter, Mage, Assassin, vb.) olma oranı*. Bu oran ne kadar yüksekse model o kadar tutarlı öneriler üretiyor demektir.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def evaluate_k(sf, herotypes, k_range, metric):
    """Her k değeri için komşuların herotype eşleşme oranını hesaplar."""
    scores = []
    for k in k_range:
        knn_tmp = NearestNeighbors(n_neighbors=k+1, metric=metric)
        knn_tmp.fit(sf)
        _, indices = knn_tmp.kneighbors(sf)
        match, total = 0, 0
        for i in range(len(sf)):
            neighbor_idx = indices[i][1:]  # ilk eleman kendisi, onu atla
            match += sum(herotypes[j] == herotypes[i] for j in neighbor_idx)
            total += len(neighbor_idx)
        scores.append(match / total)
    return scores

herotypes = df['herotype'].values
k_range = range(3, 15)

plt.figure(figsize=(7,4))
for metric in ['cosine', 'euclidean', 'manhattan']:
    scores = evaluate_k(scaled_features, herotypes, k_range, metric)
    plt.plot(list(k_range), scores, marker='o', label=metric)
    best_k = list(k_range)[int(np.argmax(scores))]
    print(f"{metric}: en iyi k={best_k}, skor={max(scores):.3f}")

plt.xlabel('k (komşu sayısı)')
plt.ylabel('Aynı herotype eşleşme oranı')
plt.title('Metrik ve k Seçiminin Karşılaştırılması')
plt.legend()
plt.tight_layout()
plt.savefig('knn_metric_comparison.png', dpi=110)
plt.show()

`manhattan` metriği genelde en yüksek tutarlılığı veriyor. Şimdi oyun tarzını belirleyen kolonlara ağırlık ekleyip aynı testi tekrarlayalım.

In [ ]:
# Oyun tarzını doğrudan tanımlayan kolonlara ekstra ağırlık veriyoruz
playstyle_cols = ['difficulty', 'damage', 'toughness', 'control', 'mobility', 'utility', 'style', 'herotype_ec']

weights = np.ones(len(feature_names))
for i, name in enumerate(feature_names):
    if name in playstyle_cols:
        weights[i] = 2.0       # şampiyonun oynanışını tanımlayan skorlar
    elif name.startswith('client_pos_'):
        weights[i] = 1.5       # pozisyon bilgisi de önemli ama biraz daha az

weighted_features = scaled_features * weights

scores_raw = evaluate_k(scaled_features, herotypes, k_range, 'manhattan')
scores_weighted = evaluate_k(weighted_features, herotypes, k_range, 'manhattan')

print(f"Ağırlıksız en iyi skor: {max(scores_raw):.3f} (k={list(k_range)[int(np.argmax(scores_raw))]})")
print(f"Ağırlıklı  en iyi skor: {max(scores_weighted):.3f} (k={list(k_range)[int(np.argmax(scores_weighted))]})")

plt.figure(figsize=(7,4))
plt.plot(list(k_range), scores_raw, marker='o', label='Ağırlıksız')
plt.plot(list(k_range), scores_weighted, marker='s', label='Ağırlıklı (oyun tarzı vurgulu)')
plt.xlabel('k (komşu sayısı)')
plt.ylabel('Aynı herotype eşleşme oranı')
plt.title('Özellik Ağırlıklandırmanın Etkisi')
plt.legend()
plt.tight_layout()
plt.savefig('knn_weight_comparison.png', dpi=110)
plt.show()

**Sonuç:** Ağırlıklandırma, komşuların aynı `herotype` olma oranını gözle görülür şekilde artırıyor. Bu, modelin artık sadece ham istatistiklere değil, şampiyonun gerçek oynanış kimliğine göre de benzerlik kurduğu anlamına gelir.

### 9.2 Final Model

In [ ]:
BEST_K = 6            # önerilecek komşu sayısı (kendisi dahil)
BEST_METRIC = 'manhattan'

knn = NearestNeighbors(n_neighbors=BEST_K, metric=BEST_METRIC)
knn.fit(weighted_features)

def find_similar_champions(champion_name, df, feats, knn_model, n=5):
    """Verilen şampiyon için en benzer n şampiyonu bulur."""
    matches = df[df['apiname'] == champion_name]
    if matches.empty:
        raise ValueError(f"'{champion_name}' isimli şampiyon bulunamadı.")
    idx = matches.index[0]
    distances, indices = knn_model.kneighbors([feats[idx]], n_neighbors=n+1)
    similar_champs = df.iloc[indices[0][1:n+1]][['apiname', 'herotype', 'role']].copy()
    similar_champs['distance'] = distances[0][1:n+1]
    return similar_champs.reset_index(drop=True)

print("Irelia için benzer şampiyonlar:")
print(find_similar_champions('Irelia', df, weighted_features, knn))
print("\nSyndra için benzer şampiyonlar:")
print(find_similar_champions('Syndra', df, weighted_features, knn))

### Düzeltme 5 — Yazım hataları
- `'irealia için...'` → `'Irelia için...'`
- `'\Syandra için...'` → `'Syndra için...'` (`\S` geçersiz bir escape karakteriydi ve isim de yanlış yazılmıştı)
- `plot_similar_championsss(...)` fonksiyon adında fazladan bir `s` vardı; tanımlanan fonksiyon adı `plot_similar_champions` idi, bu yüzden bu satır **`NameError`** ile patlıyordu. Aşağıda düzeltildi.

### 9.3 Görselleştirme

In [ ]:
def plot_similar_champions(champion_name, df, feats, knn_model, n=5):
    similar_champs = find_similar_champions(champion_name, df, feats, knn_model, n)
    plt.figure(figsize=(8, 4))
    bars = plt.bar(similar_champs['apiname'], similar_champs['distance'], color='#e67e22')
    plt.title(f"'{champion_name}' için En Benzer Şampiyonlar")
    plt.xlabel('Şampiyon')
    plt.ylabel('Uzaklık (küçük = daha benzer)')
    plt.xticks(rotation=45)
    for b in bars:
        plt.text(b.get_x()+b.get_width()/2, b.get_height(), f'{b.get_height():.2f}', ha='center', va='bottom')
    plt.tight_layout()
    plt.savefig(f'similar_{champion_name}.png', dpi=110)
    plt.show()
    return similar_champs

plot_similar_champions('Zed', df, weighted_features, knn)

Artık fonksiyon adı doğru çağrılıyor (`plot_similar_champions`, üç `s` değil) ve grafik hem konsola hem de dosyaya (`similar_<isim>.png`) kaydediliyor.